# 04 Teacher Logits

Cel: uruchomic teacher model i zapisac logitsy dla zbioru train/validation/test.


In [ ]:
# Krok 1: importy i konfiguracja
# Po co to robimy: przygotowujemy biblioteki i stale do oceny teachera, ewentualnego treningu i zapisu logits.

# Importujemy os do pracy ze sciezkami i zmiennymi srodowiskowymi.
import os
# Ustawiamy flage, aby ukryc warning o symlinkach na Windows.
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
# Ustawiamy lokalny cache HF, aby latwiej kontrolowac pliki.
os.environ.setdefault("HF_HOME", "../artifacts/hf_cache")
# Wylaczamy Xet backend, aby nie dostawac komunikatu o hf_xet.
os.environ["HF_HUB_DISABLE_XET"] = "1"

# Importujemy json do zapisu metryk teachera.
import json
# Importujemy numpy do laczenia i zapisu logits.
import numpy as np
# Importujemy PyTorch do inferencji i treningu teachera.
import torch
# Importujemy funkcje do clippingu gradientow.
from torch.nn.utils import clip_grad_norm_
# Importujemy loader danych z dysku.
from datasets import load_from_disk
# Importujemy metryki accuracy i F1 macro.
from sklearn.metrics import accuracy_score, f1_score
# Importujemy DataLoader do batchowania danych.
from torch.utils.data import DataLoader
# Importujemy pasek postepu.
from tqdm.auto import tqdm
# Importujemy model i tokenizer z Transformers.
from transformers import AutoModelForSequenceClassification, AutoTokenizer

# Ustalamy nazwe modelu teacher.
TEACHER_MODEL_NAME = "distilroberta-base"
# Ustalamy batch size dla ewaluacji i treningu.
BATCH_SIZE = 8
# Ustalamy liczbe epok opcjonalnego fine-tuningu teachera.
TEACHER_EPOCHS = 3
# Ustalamy learning rate dla teachera.
TEACHER_LR = 1e-5
# Ustalamy clipping gradientow dla teachera.
MAX_GRAD_NORM = 1.0
# Ustalamy, czy mamy trenowac teachera, gdy jest za slaby.
FINETUNE_TEACHER_IF_WEAK = True
# Ustalamy prog F1 macro, ponizej ktorego teacher jest traktowany jako slaby.
TARGET_TEACHER_TEST_F1 = 0.60
# Ustalamy urzadzenie CPU.
DEVICE = torch.device("cpu")
# Ustalamy katalog artefaktow teachera.
teacher_path = "../artifacts/teacher"


In [ ]:
# Krok 2: ladowanie danych i modelu teachera
# Po co to robimy: przygotowujemy wszystko, aby najpierw sprawdzic jakosc teachera, a dopiero potem ewentualnie go trenowac.

# Wczytujemy tokenized dane teachera z dysku.
teacher_ds = load_from_disk("../artifacts/tokenized_teacher")
# Tworzymy loader treningowy z tasowaniem.
train_loader = DataLoader(teacher_ds["train"], batch_size=BATCH_SIZE, shuffle=True)
# Tworzymy loader walidacyjny bez tasowania.
val_loader = DataLoader(teacher_ds["validation"], batch_size=BATCH_SIZE, shuffle=False)
# Tworzymy loader testowy bez tasowania.
test_loader = DataLoader(teacher_ds["test"], batch_size=BATCH_SIZE, shuffle=False)

# Sprawdzamy, czy mamy lokalny model teachera zapisany w artefaktach.
if os.path.isdir(teacher_path) and os.path.exists(os.path.join(teacher_path, "config.json")):
    # Wczytujemy model teachera z lokalnych artefaktow.
    teacher = AutoModelForSequenceClassification.from_pretrained(teacher_path)
    # Wczytujemy tokenizer teachera z lokalnych artefaktow.
    tokenizer = AutoTokenizer.from_pretrained(teacher_path)
# Jesli lokalnego modelu nie ma, ladujemy pretrained z Hub.
else:
    # Wczytujemy pretrained teacher z glowa na 3 klasy.
    teacher = AutoModelForSequenceClassification.from_pretrained(TEACHER_MODEL_NAME, num_labels=3)
    # Wczytujemy tokenizer teachera z Hub.
    tokenizer = AutoTokenizer.from_pretrained(TEACHER_MODEL_NAME)

# Przenosimy model teachera na CPU.
teacher.to(DEVICE)
# Definiujemy optymalizator dla ewentualnego fine-tuningu teachera.
optimizer = torch.optim.AdamW(teacher.parameters(), lr=TEACHER_LR)
# Definiujemy CE loss dla treningu teachera.
criterion = torch.nn.CrossEntropyLoss()


In [ ]:
# Krok 3: funkcja ewaluacji teachera
# Po co to robimy: mierzymy jak dobry jest teacher zanim uzyjemy jego logits do distillation.

# Definiujemy funkcje, ktora zwraca accuracy i F1 macro dla dowolnego loadera.
def evaluate(model, dataloader):
    # Ustawiamy tryb ewaluacji.
    model.eval()
    # Tworzymy liste na predykcje modelu.
    preds = []
    # Tworzymy liste na prawdziwe etykiety.
    labels = []
    # Wylaczamy gradienty, bo to tylko pomiar.
    with torch.no_grad():
        # Iterujemy po batchach.
        for batch in dataloader:
            # Pobieramy input_ids i przenosimy na CPU.
            input_ids = batch["input_ids"].to(DEVICE)
            # Pobieramy attention_mask i przenosimy na CPU.
            attention_mask = batch["attention_mask"].to(DEVICE)
            # Pobieramy etykiety i przenosimy na CPU.
            y = batch["label"].to(DEVICE)
            # Liczymy logits teachera.
            logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
            # Zamieniamy logits na klasy i dopisujemy do listy.
            preds.extend(torch.argmax(logits, dim=-1).cpu().tolist())
            # Dopisujemy prawdziwe etykiety do listy.
            labels.extend(y.cpu().tolist())
    # Zwracamy metryki modelu.
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro"),
    }


In [ ]:
# Krok 4: najpierw oceniamy teachera, potem opcjonalnie trenujemy
# Po co to robimy: nie chcemy slepo robic distillation od slabego teachera.

# Liczymy metryki teachera przed treningiem na walidacji.
teacher_val_before = evaluate(teacher, val_loader)
# Liczymy metryki teachera przed treningiem na tescie.
teacher_test_before = evaluate(teacher, test_loader)
# Wypisujemy metryki teachera przed treningiem.
print("Teacher BEFORE - Validation:", teacher_val_before)
print("Teacher BEFORE - Test:", teacher_test_before)

# Sprawdzamy, czy teacher jest ponizej oczekiwanego progu F1 i czy fine-tuning jest wlaczony.
should_finetune = FINETUNE_TEACHER_IF_WEAK and (teacher_test_before["f1_macro"] < TARGET_TEACHER_TEST_F1)
print("Czy uruchomic fine-tuning teachera:", should_finetune)

# Jesli teacher jest slaby, uruchamiamy fine-tuning teachera.
if should_finetune:
    # Iterujemy po epokach treningu teachera.
    for epoch in range(TEACHER_EPOCHS):
        # Ustawiamy tryb treningu.
        teacher.train()
        # Tworzymy pasek postepu treningu.
        pbar = tqdm(train_loader, desc=f"Teacher epoch {epoch + 1}/{TEACHER_EPOCHS}")
        # Iterujemy po batchach treningowych.
        for batch in pbar:
            # Pobieramy input_ids i przenosimy na CPU.
            input_ids = batch["input_ids"].to(DEVICE)
            # Pobieramy attention_mask i przenosimy na CPU.
            attention_mask = batch["attention_mask"].to(DEVICE)
            # Pobieramy etykiety i przenosimy na CPU.
            labels = batch["label"].to(DEVICE)
            # Liczymy logits teachera.
            logits = teacher(input_ids=input_ids, attention_mask=attention_mask).logits
            # Liczymy CE loss teachera.
            loss = criterion(logits, labels)
            # Liczymy gradienty.
            loss.backward()
            # Stosujemy clipping gradientow.
            clip_grad_norm_(teacher.parameters(), MAX_GRAD_NORM)
            # Aktualizujemy wagi teachera.
            optimizer.step()
            # Zerujemy gradienty przed kolejnym krokiem.
            optimizer.zero_grad(set_to_none=True)
            # Wyswietlamy loss na pasku postepu.
            pbar.set_postfix({"loss": float(loss.detach().cpu())})

# Liczymy metryki teachera po ewentualnym treningu na walidacji.
teacher_val_after = evaluate(teacher, val_loader)
# Liczymy metryki teachera po ewentualnym treningu na tescie.
teacher_test_after = evaluate(teacher, test_loader)
# Wypisujemy metryki teachera po etapie fine-tuningu.
print("Teacher AFTER - Validation:", teacher_val_after)
print("Teacher AFTER - Test:", teacher_test_after)


In [ ]:
# Krok 5: zapisujemy metryki teachera i definiujemy zapis logits
# Po co to robimy: chcemy miec slady eksperymentu i przygotowac dane do distillation.

# Tworzymy slownik z metrykami teachera przed i po fine-tuningu.
teacher_metrics = {
    "before": {"validation": teacher_val_before, "test": teacher_test_before},
    "after": {"validation": teacher_val_after, "test": teacher_test_after},
    "finetuned": bool(should_finetune),
}
# Zapisujemy metryki teachera do pliku JSON.
with open("../outputs/metrics_teacher.json", "w", encoding="utf-8") as f:
    json.dump(teacher_metrics, f, indent=2)
print("Zapisano: ../outputs/metrics_teacher.json")

# Definiujemy funkcje zapisu logits dla wybranego splitu.
def save_split_logits(split_name):
    # Tworzymy loader splitu bez tasowania.
    loader = DataLoader(teacher_ds[split_name], batch_size=BATCH_SIZE, shuffle=False)
    # Tworzymy liste na indeksy rekordow.
    all_idx = []
    # Tworzymy liste na logits batchy.
    all_logits = []
    # Tworzymy liste na etykiety.
    all_labels = []
    # Wylaczamy gradienty dla inferencji.
    with torch.no_grad():
        # Ustawiamy teacher w tryb ewaluacji.
        teacher.eval()
        # Iterujemy po batchach i liczymy logits.
        for batch in tqdm(loader, desc=f"Teacher logits: {split_name}"):
            # Pobieramy input_ids i przenosimy na CPU.
            input_ids = batch["input_ids"].to(DEVICE)
            # Pobieramy attention_mask i przenosimy na CPU.
            attention_mask = batch["attention_mask"].to(DEVICE)
            # Liczymy logits teachera.
            logits = teacher(input_ids=input_ids, attention_mask=attention_mask).logits
            # Dopisujemy indeksy rekordow.
            all_idx.extend(batch["idx"].cpu().tolist())
            # Dopisujemy logits do listy.
            all_logits.append(logits.cpu().numpy())
            # Dopisujemy etykiety do listy.
            all_labels.extend(batch["label"].cpu().tolist())

    # Laczymy logits ze wszystkich batchy w jedna tablice.
    logits_np = np.concatenate(all_logits, axis=0)
    # Budujemy sciezke zapisu pliku logits.
    save_path = f"../artifacts/teacher_logits_{split_name}.npz"
    # Zapisujemy idx, logits i labels do pliku npz.
    np.savez(save_path, idx=np.array(all_idx), logits=logits_np, labels=np.array(all_labels))
    # Potwierdzamy zapis pliku.
    print(f"Zapisano: {save_path}")


In [ ]:
# Krok 6: zapisujemy logits i finalny model teachera
# Po co to robimy: przygotowujemy miekkie etykiety dla distillation i utrwalamy finalna wersje teachera.

# Iterujemy po wszystkich splitach i zapisujemy logits.
for split in ["train", "validation", "test"]:
    # Liczymy i zapisujemy logits dla biezacego splitu.
    save_split_logits(split)

# Tworzymy katalog artefaktow teachera, jesli jeszcze nie istnieje.
os.makedirs(teacher_path, exist_ok=True)
# Ustalamy osobny katalog finalny, aby nie nadpisywac plikow mapowanych z poprzedniego wczytania.
teacher_final_path = "../artifacts/teacher_finetuned"
# Tworzymy katalog docelowy dla finalnego teachera.
os.makedirs(teacher_final_path, exist_ok=True)

# Zapisujemy finalny model teachera bez safetensors, aby ominac blad Windows os error 1224.
teacher.save_pretrained(teacher_final_path, safe_serialization=False)
# Zapisujemy finalny tokenizer teachera.
tokenizer.save_pretrained(teacher_final_path)

# Zapisujemy plik wskazujacy, gdzie jest finalny model teachera.
with open("../outputs/teacher_final_path.txt", "w", encoding="utf-8") as f:
    f.write(teacher_final_path + "\n")

# Potwierdzamy zapis modelu i tokenizera teachera.
print("Zapisano teacher model i tokenizer do:", teacher_final_path)
